In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pickle
import os
import subprocess
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import compare_prices
from e_2_CVAE import *


# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
opt_type = 'call' # call or put
barr_type = 'van' # van or barr
model_type = 'bs' # hes or bs

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# training

In [30]:
import os, subprocess, sys

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"

code = """
import torch, os
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("device_count =", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
"""

subprocess.run([sys.executable, "-c", code], env=env, check=True)

CUDA_VISIBLE_DEVICES = 0,1,2,3,4,5,6,7
device_count = 7
0 NVIDIA GeForce GTX 1080 Ti
1 NVIDIA GeForce GTX 1080 Ti
2 NVIDIA GeForce GTX 1080 Ti
3 NVIDIA GeForce GTX 1080 Ti
4 NVIDIA GeForce GTX 1080 Ti
5 NVIDIA GeForce GTX 1080 Ti
6 NVIDIA GeForce GTX 1080 Ti


/home/ajoufe/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:497: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


CompletedProcess(args=['/home/ajoufe/anaconda3/bin/python', '-c', '\nimport torch, os\nprint("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))\nprint("device_count =", torch.cuda.device_count())\nfor i in range(torch.cuda.device_count()):\n    print(i, torch.cuda.get_device_name(i))\n'], returncode=0)

In [ ]:
# CVAE DDP training settings
dim_z       = 8 # 12
hidden_dims = [512, 512, 256] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096
lr          = 1e-3 # 3e-4, 5e-4
beta        = 1.0
use_bn = False
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size*7}_chunk{num_chunks}_NBN_DDP.pt"
load_path = None # 과거 weight를 저장하지 않았던 모델에 적용하기 위한 변수
resume_path = None # 이어서 학습하고 싶을 때

n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

gpu_ids = "0,1,2,3,4,5,6" # 7 GPU problem
nproc = len(gpu_ids.split(","))

hidden_dims_arg = ",".join(map(str, hidden_dims))


In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--barr-type", barr_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if use_bn:
    cmd += ["--use-bn"]
if load_path is not None:
    cmd += ["--load-path", str(load_path)]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


In [ ]:
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk10_NBN_DDP.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk5_NBN_DDP.pt"

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--barr-type", barr_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if use_bn:
    cmd += ["--use-bn"]
if load_path is not None:
    cmd += ["--load-path", str(load_path)]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


In [ ]:
num_chunks = 10
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk20_NBN_DDP.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk10_NBN_DDP.pt"

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--barr-type", barr_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if use_bn:
    cmd += ["--use-bn"]
if load_path is not None:
    cmd += ["--load-path", str(load_path)]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


# use BN

In [ ]:
use_bn = True
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk5_BN_DDP.pt"
resume_path = None

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--barr-type", barr_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if use_bn:
    cmd += ["--use-bn"]
if load_path is not None:
    cmd += ["--load-path", str(load_path)]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


In [ ]:
num_chunks = 5
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk10_BN_DDP.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk5_BN_DDP.pt"

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--barr-type", barr_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if use_bn:
    cmd += ["--use-bn"]
if load_path is not None:
    cmd += ["--load-path", str(load_path)]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


In [ ]:
num_chunks = 10
save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk20_BN_DDP.pt"
resume_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_chunk10_BN_DDP.pt"

In [ ]:
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = gpu_ids
env["NCCL_DEBUG"] = "INFO"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"

cmd = [
    "torchrun",
    f"--nproc_per_node={nproc}",
    "f_run_cvae_ddp.py",
    "--model-type", model_type,
    "--barr-type", barr_type,
    "--dim-z", str(dim_z),
    "--hidden-dims", hidden_dims_arg,
    "--batch-size", str(batch_size),
    "--num-chunks", str(num_chunks),
    "--lr", str(lr),
    "--beta", str(beta),
    "--save-path", str(save_path),
    "--num-workers", "8",
    "--prefetch-factor", "2",
]

if use_bn:
    cmd += ["--use-bn"]
if load_path is not None:
    cmd += ["--load-path", str(load_path)]
if resume_path is not None:
    cmd += ["--resume-path", str(resume_path)]

time1 = time.time()
subprocess.run(cmd, env=env, check=True)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")


# compare distribution of X_T and M_T

In [ ]:
# real X, M
real_x, real_m = load_real_xt_mt_from_chunks(
    chunk_dir="/mnt/d/bs_chunks_correction/",
    num_chunks=10,
)

statics_result("Real X_T", real_x)
if len(real_m) != 0:
    statics_result("Real M_T", real_m)

plot_1d("Real X_T", real_x, bins=100)
if len(real_m) != 0:
    plot_1d("Real M_T", real_m, bins=100)
    plot_2d_density(
        real_x, real_m,
        title="Real MC",
        xlabel="X_T",
        ylabel="M_T",
        bins=100,
    )

In [ ]:
# use BN
x_bn, m_bn, ckpt_bn = load_cvae_xt_mt(
    save_path=save_path_bn,
    test_etas=test_etas,
    n_samples=len(real_x),
)

statics_result("CVAE BN X_T", x_bn)
if len(m_bn) != 0:
    statics_result("CVAE BN M_T", m_bn)


plot_1d("CVAE BN X_T", x_bn, bins=100)
if len(m_bn) != 0:
    plot_1d("CVAE BN M_T", m_bn, bins=100)
    plot_2d_density(
        x_bn, m_bn,
        title="CVAE BN",
        xlabel="X_T",
        ylabel="M_T",
        bins=100,
    )


In [ ]:
# not use BN
x_no_bn, m_no_bn, ckpt_no_bn = load_cvae_xt_mt(
    save_path=save_path_no_bn,
    test_etas=test_etas,
    n_samples=len(real_x),
)

statics_result("CVAE no BN X_T", x_no_bn)
if len(m_no_bn) != 0:
    statics_result("CVAE no BN M_T", m_no_bn)

plot_1d("CVAE no BN X_T", x_no_bn, bins=100)
if len(m_no_bn) != 0:
    plot_1d("CVAE no BN M_T", m_no_bn, bins=100)
    plot_2d_density(
        x_no_bn, m_no_bn,
        title="CVAE no BN",
        xlabel="X_T",
        ylabel="M_T",
        bins=100,
    )


In [ ]:
# vanilla X_t
plot_three_distributions(real_x, x_no_bn, x_bn, name="X_T", bins=100)

real_x, real_m = load_real_xt_mt_from_chunks(chunk_dir, num_chunks=10)
x_no_bn, m_no_bn, _ = load_cvae_xt_mt(save_path_no_bn, test_etas, n_samples=len(real_x))
x_bn, m_bn, _ = load_cvae_xt_mt(save_path_bn, test_etas, n_samples=len(real_x))

real_price = vanilla_price_from_xt(real_x, K, r, T, opt_type)
no_bn_price = vanilla_price_from_xt(x_no_bn, K, r, T, opt_type)
bn_price = vanilla_price_from_xt(x_bn, K, r, T, opt_type)

print("Real vanilla price    :", real_price)
print("No BN price   :", no_bn_price, f"error={price_error(no_bn_price, real_price):+.2f}%")
print("BN price      :", bn_price, f"error={price_error(bn_price, real_price):+.2f}%")

In [ ]:
# barrier M_t
plot_three_distributions(real_m, m_no_bn, m_bn, name="M_T", bins=100)

real_x, real_m = load_real_xt_mt_from_chunks(chunk_dir, num_chunks=10)
x_no_bn, m_no_bn, _ = load_cvae_xt_mt(save_path_no_bn, test_etas, n_samples=len(real_x))
x_bn, m_bn, _ = load_cvae_xt_mt(save_path_bn, test_etas, n_samples=len(real_x))

real_price = barrier_price_from_xt_mt(real_x, real_m, K, r, T, opt_type)
no_bn_price = barrier_price_from_xt_mt(x_no_bn, m_no_bn, K, r, T, opt_type)
bn_price = barrier_price_from_xt_mt(x_bn, m_bn, K, r, T, opt_type)

print("Real barrier price    :", real_price)
print("No BN price   :", no_bn_price, f"error={price_error(no_bn_price, real_price):+.2f}%")
print("BN price      :", bn_price, f"error={price_error(bn_price, real_price):+.2f}%")

# inference

In [ ]:
cvae_save_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_epoch10_NBN.pt" # = save_path

cvae_result_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_epoch10_NBN.pkl"

ckpt = torch.load(cvae_save_path, map_location=device, weights_only=False)

cvae = CVAE(
    dim_x=ckpt["dim_x"],
    dim_eta=ckpt["dim_eta"],
    dim_z=ckpt["dim_z"],
    hidden_dims=ckpt["hidden_dims"],
    use_bn=ckpt.get("use_bn", False),
).to(device)

state_dict = ckpt["model_state"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

cvae.load_state_dict(state_dict)
cvae.eval()

eta_raw = np.array(test_etas, dtype=np.float32)
eta_scaled = (eta_raw - ckpt["eta_min"]) / (ckpt["eta_max"] - ckpt["eta_min"] + 1e-8)
eta_t = torch.tensor(eta_scaled, dtype=torch.float32, device=device)

n_list = [1000, 10000, 100000]
n_repeats = 50
results = {n: [] for n in n_list}


for n in n_list:
    for _ in range(n_repeats):
        if barr_type == "barr":
            price = cvae.price_barrier(eta_t, B, K, r, T, opt_type, n)
        else:
            price = cvae.price_vanilla(eta_t, K, r, T, opt_type, n)
        results[n].append(price)

with open(cvae_result_path, "wb") as f:
    pickle.dump(results, f)

In [ ]:
# 공용 그래프: CVAE inference 결과만 표시
n_list = [1000, 10000, 100000]
cvae_result_path = f"cvae_{model_type}_{barr_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_epoch5_NBN.pkl"

with open(cvae_result_path, "rb") as f:
    results = pickle.load(f)

means = np.array([np.mean(results[n]) for n in n_list])
stds = np.array([np.std(results[n]) for n in n_list])
ci = 1.96 * stds

print("CVAE")
for n, mean, err in zip(n_list, means, ci):
    rel_err = (mean - bench_price) / (bench_price + 1e-12) * 100
    print(f"n={n:6d} | mean={mean:.8f} | 95% CI=±{err:.8f} | error={rel_err:+.2f}%")

plt.figure(figsize=(7, 4))
plt.axhline(bench_price, color="gray", linewidth=1.5, label="FDM")
plt.errorbar(
    range(len(n_list)), means,
    yerr=ci, fmt="o-", color="steelblue", capsize=5, label="CVAE (95% CI)",
)
plt.xticks(range(len(n_list)), ["1K", "10K", "100K"])
plt.xlabel("Number of CVAE samples")
plt.ylabel("Option price")
plt.title("CVAE convergence")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
